# 🤖 Polymarket Trading Bot - Google Colab

Este notebook te permite ejecutar el bot de trading de Polymarket en Google Colab de forma **GRATUITA**.

## ⚠️ Importante
- Este notebook está configurado para **PAPER TRADING** (simulación)
- No se usará dinero real
- Necesitas API keys de Gemini y Groq (ambas GRATIS)
- El bot se ejecutará mientras el notebook esté activo (máx 12 horas)

## 📋 Pasos
1. Ejecuta cada celda en orden (Shift + Enter)
2. Configura tus API keys cuando se solicite
3. El bot empezará a funcionar automáticamente
4. Revisa los logs y el dashboard

## 1️⃣ Clonar el Repositorio

In [ ]:
# Clonar el código del bot
!git clone https://github.com/TU_USUARIO/polymarket-trading-bot.git
%cd polymarket-trading-bot

# Mostrar estructura
!ls -la

## 2️⃣ Instalar Dependencias

In [ ]:
# Instalar todas las dependencias
!pip install -q -r requirements.txt

print("✅ Dependencias instaladas correctamente")

## 3️⃣ Configurar API Keys

### Obtener API Keys (GRATIS):
1. **Gemini**: https://aistudio.google.com/app/apikey
2. **Groq**: https://console.groq.com/keys

### Para Paper Trading:
- Las API keys de Polymarket son **OPCIONALES** en modo paper trading
- Solo necesitas Gemini o Groq para las predicciones

In [ ]:
import os
from getpass import getpass

# Solicitar API keys de forma segura
print("🔑 Configuración de API Keys")
print("="*50)

# Gemini (REQUERIDO)
gemini_key = getpass("Ingresa tu Gemini API Key: ")
os.environ['GEMINI_API_KEY'] = gemini_key

# Groq (OPCIONAL pero recomendado como backup)
use_groq = input("¿Tienes Groq API Key? (s/n): ").lower()
if use_groq == 's':
    groq_key = getpass("Ingresa tu Groq API Key: ")
    os.environ['GROQ_API_KEY'] = groq_key
else:
    os.environ['GROQ_API_KEY'] = ''

# NewsAPI (OPCIONAL)
use_news = input("¿Tienes NewsAPI Key? (s/n): ").lower()
if use_news == 's':
    news_key = getpass("Ingresa tu NewsAPI Key: ")
    os.environ['NEWS_API_KEY'] = news_key
else:
    os.environ['NEWS_API_KEY'] = ''

# Configuración de Paper Trading
os.environ['TRADING_MODE'] = 'paper'
os.environ['INITIAL_BANKROLL'] = '100'
os.environ['DATABASE_URL'] = 'sqlite:///data/trades.db'
os.environ['LOG_LEVEL'] = 'INFO'

# Polymarket (vacío para paper trading)
os.environ['POLYMARKET_API_KEY'] = ''
os.environ['POLYMARKET_API_SECRET'] = ''
os.environ['POLYMARKET_API_PASSPHRASE'] = ''
os.environ['POLYMARKET_PRIVATE_KEY'] = ''

print("\n✅ API Keys configuradas correctamente")
print("📊 Modo: PAPER TRADING (simulación)")
print("💰 Bankroll inicial: $100 USDC (virtual)")

## 4️⃣ Crear Directorios Necesarios

In [ ]:
# Crear directorios para datos y logs
!mkdir -p data logs

print("✅ Directorios creados")

## 5️⃣ Verificar Configuración

In [ ]:
# Verificar que todo está configurado correctamente
from src.utils.config import get_config

config = get_config()

print("🔍 Verificación de Configuración")
print("="*50)
print(f"Modo de Trading: {config.trading_mode.upper()}")
print(f"Bankroll Inicial: ${config.trading.initial_bankroll}")
print(f"Kelly Fraction: {config.trading.kelly_fraction}")
print(f"Max Risk/Trade: {config.trading.max_risk_per_trade:.1%}")
print()

# Validar API keys
validations = config.validate_api_keys()
print("API Keys:")
for api, is_valid in validations.items():
    status = "✅" if is_valid else "❌"
    print(f"{status} {api}: {'Configurada' if is_valid else 'No configurada'}")

if validations['gemini'] or validations['groq']:
    print("\n✅ Configuración válida - Listo para ejecutar")
else:
    print("\n❌ ERROR: Necesitas al menos Gemini o Groq API key")

## 6️⃣ Ejecutar el Bot (Modo Una Vez)

Esta celda ejecutará el bot **una vez** para probar que funciona.

El bot:
1. Escaneará mercados (simulados en paper trading)
2. Generará predicciones con LLM
3. Ejecutará trades virtuales si encuentra oportunidades
4. Guardará todo en la base de datos

In [ ]:
# Ejecutar el bot una vez
from main import TradingBot

print("🚀 Iniciando bot...\n")

bot = TradingBot()
bot.run_once()

print("\n✅ Ejecución completada")

## 7️⃣ Ver Resultados

In [ ]:
# Ver trades ejecutados
from src.models.database import get_database, Trade
import pandas as pd

db = get_database()
session = db.get_session()

# Obtener todos los trades
trades = session.query(Trade).all()

if trades:
    trade_data = []
    for t in trades:
        trade_data.append({
            'ID': t.id,
            'Market': t.market_question[:50] + '...',
            'Side': f"{t.side} {t.outcome}",
            'Size': f"${t.size:.2f}",
            'Entry': f"{t.entry_price:.3f}",
            'Status': t.status,
            'P&L': f"${t.realized_pnl or t.unrealized_pnl or 0:.2f}",
            'Time': t.entry_time.strftime('%Y-%m-%d %H:%M')
        })
    
    df = pd.DataFrame(trade_data)
    print("📊 Trades Ejecutados:")
    print(df.to_string(index=False))
else:
    print("ℹ️ No se ejecutaron trades en esta ejecución")
    print("Esto es normal si no se encontraron oportunidades que cumplan los criterios")

session.close()

## 8️⃣ Ver Predicciones del LLM

In [ ]:
# Ver predicciones generadas
from src.models.database import Prediction

session = db.get_session()
predictions = session.query(Prediction).order_by(Prediction.created_at.desc()).limit(5).all()

if predictions:
    print("🤖 Últimas Predicciones del LLM:")
    print("="*80)
    
    for i, pred in enumerate(predictions, 1):
        print(f"\n{i}. {pred.market_question}")
        print(f"   Predicción: {pred.predicted_outcome.upper()}")
        print(f"   Confianza: {pred.confidence}%")
        print(f"   Probabilidad Estimada: {pred.expected_probability:.1%}")
        print(f"   Precio de Mercado: {pred.current_yes_price:.3f}")
        print(f"   LLM: {pred.llm_provider.title()} ({pred.llm_model})")
        print(f"   Razonamiento: {pred.reasoning[:200]}...")
else:
    print("ℹ️ No se generaron predicciones")

session.close()

## 9️⃣ Ver Logs

In [ ]:
# Ver últimas líneas del log
!tail -50 logs/bot.log

## 🔟 Ejecutar Bot en Modo Continuo (Opcional)

⚠️ **IMPORTANTE**: Esta celda ejecutará el bot continuamente hasta que:
- Detengas manualmente la celda (botón Stop)
- Google Colab se desconecte (después de ~12 horas)

El bot:
- Escaneará mercados cada 6 horas
- Revisará posiciones cada 30 minutos
- Mostrará logs en tiempo real

In [ ]:
# SOLO ejecuta esta celda si quieres que el bot corra continuamente
from main import TradingBot

print("🚀 Iniciando bot en modo continuo...")
print("⚠️ Para detener: Click en el botón STOP o Kernel > Interrupt")
print("="*80)

bot = TradingBot()
bot.run_continuous()

## 📊 Dashboard (Alternativa)

Para ver el dashboard de Streamlit en Colab, necesitamos usar ngrok o localtunnel.
Aquí una versión simplificada con visualizaciones:

In [ ]:
# Visualización simple del rendimiento
import matplotlib.pyplot as plt
from src.models.database import get_database, Trade

db = get_database()
session = db.get_session()

# Obtener trades cerrados
closed_trades = session.query(Trade).filter(Trade.status == 'closed').order_by(Trade.exit_time).all()

if closed_trades:
    # Calcular equity curve
    initial_bankroll = 100
    equity = [initial_bankroll]
    dates = []
    
    for trade in closed_trades:
        equity.append(equity[-1] + (trade.realized_pnl or 0))
        dates.append(trade.exit_time)
    
    # Graficar
    plt.figure(figsize=(12, 6))
    plt.plot(dates, equity[1:], marker='o', linewidth=2, markersize=8)
    plt.axhline(y=initial_bankroll, color='r', linestyle='--', label='Initial Bankroll')
    plt.title('Equity Curve - Paper Trading', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Bankroll (USDC)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    total_pnl = sum(t.realized_pnl or 0 for t in closed_trades)
    winning = len([t for t in closed_trades if (t.realized_pnl or 0) > 0])
    win_rate = (winning / len(closed_trades) * 100) if closed_trades else 0
    
    print(f"\n📊 Estadísticas:")
    print(f"Total Trades: {len(closed_trades)}")
    print(f"Winning Trades: {winning}")
    print(f"Win Rate: {win_rate:.1f}%")
    print(f"Total P&L: ${total_pnl:.2f}")
    print(f"Final Bankroll: ${equity[-1]:.2f}")
    print(f"Return: {(total_pnl/initial_bankroll)*100:.1f}%")
else:
    print("ℹ️ No hay trades cerrados para graficar")

session.close()

## 💾 Descargar Datos

Descarga la base de datos para analizarla localmente o guardar backup:

In [ ]:
from google.colab import files

# Descargar base de datos
files.download('data/trades.db')

# Descargar logs
files.download('logs/bot.log')
files.download('logs/trades.log')

print("✅ Archivos descargados")

## 🎯 Próximos Pasos

1. **Prueba el bot** ejecutando las celdas 6-8 varias veces
2. **Revisa las predicciones** del LLM y su razonamiento
3. **Ajusta la configuración** en `config/config.yaml` si es necesario
4. **Cuando estés listo para 24/7**: Migra a Railway.app
5. **Para dinero real**: Configura las API keys de Polymarket

## ⚠️ Limitaciones de Google Colab

- Máximo 12 horas de ejecución continua
- Se desconecta si está inactivo
- No es ideal para trading 24/7
- Perfecto para pruebas y paper trading

## 📚 Recursos

- [README.md](README.md) - Documentación completa
- [docs/api_setup_guide.md](docs/api_setup_guide.md) - Guía de API keys
- [docs/deployment_pythonanywhere.md](docs/deployment_pythonanywhere.md) - Deploy en producción

---

**¡Buena suerte con el trading! 🚀📈**